# Churn model — exploration

TODO: clean this up before the review (written 8 months ago)

In [7]:
# had to install these, kernel didn't have them
!pip install pandas scikit-learn numpy

In [2]:
import sys
sys.path.append("../..")
sys.path.append("/Users/rgupta/work/churn/helpers")

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [12]:
# synthetic customers (stand-in for the warehouse query)
rng = np.random.default_rng(0)
n = 8000
tenure_months = rng.integers(1, 72, size=n)
monthly_charge = rng.normal(70, 25, size=n).clip(15, 160)
support_calls = rng.poisson(1.2, size=n)
is_month_to_month = rng.binomial(1, 0.55, size=n)

logit = (-1.1 - 0.045 * tenure_months + 0.021 * monthly_charge
         + 0.38 * support_calls + 1.05 * is_month_to_month)
churned = rng.binomial(1, 1 / (1 + np.exp(-logit)))

df = pd.DataFrame({
    "tenure_months": tenure_months,
    "monthly_charge": monthly_charge,
    "support_calls": support_calls,
    "is_month_to_month": is_month_to_month,
    "churned": churned,
})
df.head()

In [3]:
# features
X = df.drop(columns=["churned"])
X["spend_to_date"] = X["monthly_charge"] * X["tenure_months"]
X["calls_per_year"] = X["support_calls"] / (X["tenure_months"] / 12)
y = df["churned"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

m = HistGradientBoostingClassifier(max_iter=120, random_state=0)
m.fit(X_train, y_train)
auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
print(auc)

0.7506027742749054


### v2 — trying more iterations

In [4]:
# NOTE: this cell was re-run after editing the features cell above,
# so the number below does not match any state you can reconstruct
m = HistGradientBoostingClassifier(max_iter=400, random_state=0)
m.fit(X_train, y_train)
print(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

0.7431905886683312


In [9]:
# copy of the feature code, but slightly different -- which one made the model above?
X2 = df.drop(columns=["churned"])
X2["spend_to_date"] = X2["monthly_charge"] * X2["tenure_months"]
X2["calls_per_yr"] = X2["support_calls"] / X2["tenure_months"] * 12

In [16]:
import pickle
pickle.dump(m, open("model.pkl", "wb"))   # ship it